# Solutions · 05 · The Nyquist criterion

Worked solutions to the exercises of [notebook 05](../notebooks/05_the_nyquist_criterion.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from englincontrol import systems as sy, stability as st, frequency as fr, polynomials as pl, design as dg
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Apply the criterion to L = K/(s - 1)^2 for K = 0.5, 2 and 5 and compare with the closed-loop poles. Is there any K that stabilises it? Then add a zero, L = K (s + 1)/(s - 1)^2, and find the stabilising range.

In [2]:
for K in (0.5, 2.0, 5.0):
    L = sy.tf([K], pl.from_roots([1.0, 1.0]))
    r = st.nyquist_criterion(L)
    print(f"K/(s - 1)^2, K = {K}: N_cw = {r['N_clockwise']}, P = {r['P']}, Z = {r['Z']}; closed-loop poles {np.round(sy.feedback(L).poles(), 3)}")
print("the Nyquist curve of K/(s-1)^2 starts at +K on the real axis and never encircles -1 counterclockwise twice: no K works")
for K in (1.0, 3.0, 10.0):
    L = sy.tf(K * np.array([1.0, 1.0]), pl.from_roots([1.0, 1.0]))
    r = st.nyquist_criterion(L)
    print(f"K (s + 1)/(s - 1)^2, K = {K}: Z = {r['Z']} (N_cw {r['N_clockwise']}); closed-loop poles {np.round(sy.feedback(L).poles(), 3)}")
print("Routh on s^2 + (K - 2) s + (K + 1): stable for K > 2")

K/(s - 1)^2, K = 0.5: N_cw = 0, P = 2, Z = 2; closed-loop poles [1.+0.707j 1.-0.707j]
K/(s - 1)^2, K = 2.0: N_cw = 0, P = 2, Z = 2; closed-loop poles [1.+1.414j 1.-1.414j]
K/(s - 1)^2, K = 5.0: N_cw = 0, P = 2, Z = 2; closed-loop poles [1.+2.236j 1.-2.236j]
the Nyquist curve of K/(s-1)^2 starts at +K on the real axis and never encircles -1 counterclockwise twice: no K works
K (s + 1)/(s - 1)^2, K = 1.0: Z = 2 (N_cw 0); closed-loop poles [0.5+1.323j 0.5-1.323j]
K (s + 1)/(s - 1)^2, K = 3.0: Z = 0 (N_cw -2); closed-loop poles [-0.5+1.936j -0.5-1.936j]
K (s + 1)/(s - 1)^2, K = 10.0: Z = 0 (N_cw -2); closed-loop poles [-6.236 -1.764]
Routh on s^2 + (K - 2) s + (K + 1): stable for K > 2


A double unstable pole needs two counterclockwise encirclements of −1, which the curve of K/(s − 1)² (which
starts at +K and sweeps through the lower half-plane) never provides: every gain leaves Z = 2 or Z = 1.
Adding a zero at −1 turns the phase around; for K > 2 the curve encircles −1 twice counterclockwise and Z = 0
- the Routh array on s² + (K − 2)s + (K + 1) gives the same threshold.

## Exercise 2

For P = 1/(s + 1)^3 with proportional gain K, find from the Nyquist curve the gain at which the loop goes unstable, and compare with the Routh limit and with the ultimate gain used in notebook 06.

In [3]:
P = sy.tf([1.0], [1.0, 3.0, 3.0, 1.0])
w = np.logspace(-2, 2, 20000)
Pj = fr.freqresp(P, w)
i = np.argmin(np.abs(np.degrees(np.unwrap(np.angle(Pj))) + 180))
K_nyq = 1 / abs(Pj[i])
print(f"Nyquist: the curve crosses the negative real axis at w = {w[i]:.4f} with |P| = {abs(Pj[i]):.5f} -> the loop reaches -1 at K = {K_nyq:.3f}")
print(f"Routh on s^3 + 3 s^2 + 3 s + 1 + K: stable iff 3*3 > 1 + K, i.e. K < 8; ultimate gain of notebook 06: {dg.ultimate_gain(P)['Ku']:.3f}")

Nyquist: the curve crosses the negative real axis at w = 1.7319 with |P| = 0.12503 -> the loop reaches -1 at K = 7.998
Routh on s^3 + 3 s^2 + 3 s + 1 + K: stable iff 3*3 > 1 + K, i.e. K < 8; ultimate gain of notebook 06: 8.000


Three statements of one fact: the Nyquist curve reaches −1 at K = 8, the Routh array goes marginal at K = 8,
and the ultimate gain of the relay/ZN method is 8. The phase crossover at √3 rad/s is where (s + 1)³ has
phase −180° - three poles each contributing 60°.

## Exercise 3

**Implement it yourself:** build the Nyquist contour for L = 1/(s^2 (s + 1)) with the indentation around the double pole, evaluate L along it and count the encirclements of -1; compare with `stability.nyquist_criterion` and with the closed-loop poles.

In [4]:
L = sy.tf([1.0], [1.0, 1.0, 0.0, 0.0])
eps, W = 1e-3, 1e4
parts = []
parts.append(1j * -np.logspace(np.log10(W), np.log10(eps), 4000))                       # down the negative axis to -j eps
th = np.linspace(-np.pi / 2, np.pi / 2, 400); parts.append(eps * np.exp(1j * th))        # the indentation around the double pole
parts.append(1j * np.logspace(np.log10(eps), np.log10(W), 4000))                         # up the positive axis
s = np.concatenate(parts)
Ls = pl.evaluate(L.num, s) / pl.evaluate(L.den, s)
phase = np.unwrap(np.angle(1 + Ls))
N_cw = -int(np.round((phase[-1] - phase[0]) / (2 * np.pi)))
print(f"encirclements by hand: N_cw = {N_cw}; library: {st.nyquist_criterion(L)['N_clockwise']}; closed-loop RHP poles: {int(np.sum(sy.feedback(L).poles().real > 0))}")

encirclements by hand: N_cw = 2; library: 2; closed-loop RHP poles: 2


The double pole at the origin is excluded by the small semicircle, whose image is a large arc that sweeps
360° (twice the angle of a simple pole), and the count comes out as two clockwise encirclements: the loop
1/(s²(s + 1)) is unstable under unity feedback with two unstable closed-loop poles, as the characteristic
polynomial confirms.